# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout
  
El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd

In [2]:
# cargar archivos
orders = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv")
catalog = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv")
marketing = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv")

In [3]:
# explorar datasets
orders.info()
orders.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [4]:
print("Filas duplicadas:", orders.duplicated().sum())
orders[orders.duplicated(keep=False)].sort_values(by='id_pedido').head(10)

Filas duplicadas: 100


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
25023,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
10082,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
25037,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
10709,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
25065,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08
10829,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08
10856,order_10856,user_1899,2025-03-18,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,1.0,72.53,0.0,72.53
25044,order_10856,user_1899,2025-03-18,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,1.0,72.53,0.0,72.53
11449,order_11449,user_1480,2025-03-21,Mexico,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,60.55,5.0,55.55
25083,order_11449,user_1480,2025-03-21,Mexico,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,60.55,5.0,55.55


In [5]:
orders.duplicated(subset="id_pedido").sum()

100

In [6]:
catalog.info()
catalog.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [7]:
print("Filas duplicadas:", catalog.duplicated().sum())

Filas duplicadas: 0


In [8]:
marketing.info()
marketing.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [9]:
print("Filas duplicadas:", marketing.duplicated().sum())

Filas duplicadas: 0


In [10]:
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [11]:
catalog.describe()

,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


In [12]:
marketing.describe()

,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [13]:
orders = orders.drop_duplicates(keep='first').reset_index(drop=True)

In [14]:
orders.duplicated().sum()

0

<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
    Se eliminaron 100 filas duplicadas de orders (0.4% de las 25,100 originales). Eran copias exactas de otros pedidos (mismo id_pedido y mismos valores en todas las columnas), y en las filas revisadas aparecían al final del dataset, lo que sugiere un error de carga y no pedidos reales. Conservarlas habría inflado el número de pedidos y el revenue, y distorsionado el funnel y las cohortes. Se conservó la primera aparición de cada pedido y el dataset quedó con 25,000 filas.
</div>  

In [15]:
print("Categoría Prod en Orders:",orders["categoria_producto"].unique())
print("Categoría Prod en Catalog:",catalog["categoria_producto"].unique())

Categoría Prod en Orders: ['Moda' 'Electronica' 'Hogar' nan]
Categoría Prod en Catalog: ['Electrónica' 'Hogar' 'Moda']


In [16]:
orders["categoria_producto"]= orders["categoria_producto"].replace("Electronica","Electrónica")

In [17]:
orders["categoria_producto"].isna().sum()

80

In [18]:
mapa_categoria = catalog.set_index("nombre_producto")["categoria_producto"].to_dict()

In [19]:
orders["categoria_producto"] = orders["categoria_producto"].fillna(orders["nombre_producto"].map(mapa_categoria))

In [20]:
orders["categoria_producto"].isna().sum()

30

In [21]:
orders[orders["categoria_producto"].isna()]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
44,order_44,user_2899,2025-02-22,Colombia,desktop,NaN,NaN,NaN,1.0,318.38,5.0,313.38
45,order_45,user_6196,2025-02-10,Mexico,mobile,NaN,NaN,NaN,2.0,354.06,10.0,698.12
46,order_46,user_5815,2025-04-24,Mexico,mobile,NaN,NaN,NaN,1.0,359.31,10.0,349.31
47,order_47,user_406,2025-05-08,Colombia,mobile,NaN,NaN,NaN,2.0,476.78,0.0,953.56
48,order_48,user_6187,2025-01-26,Colombia,desktop,NaN,NaN,NaN,2.0,137.96,0.0,275.93
49,order_49,user_7225,2025-04-04,Argentina,desktop,NaN,NaN,NaN,2.0,433.76,0.0,867.53
50,order_50,user_1840,2025-02-05,Mexico,desktop,NaN,NaN,NaN,1.0,495.38,5.0,490.38
51,order_51,user_5998,2025-02-21,Mexico,desktop,NaN,NaN,NaN,2.0,382.94,0.0,765.89
52,order_52,user_5679,2025-04-17,Argentina,mobile,NaN,NaN,NaN,2.0,415.01,0.0,830.01
53,order_53,user_5936,2025-01-02,Argentina,desktop,NaN,NaN,NaN,2.0,236.49,0.0,472.98


<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
    Se unificó la escritura de la categoría Electrónica en orders (aparecía como “Electronica”, sin acento), para que coincidiera con la del catálogo y no se contara como dos categorías distintas. Además, se detectaron 80 nulos en categoria_producto (0.3% de las filas). Como cada producto pertenece a una sola categoría en el catálogo, se creó un diccionario producto → categoría (mapa_categoria) y se usó con .fillna() para rellenar solo los vacíos, sin tocar los valores existentes. 
    Se recuperaron 50 filas. Las 30 restantes no se pudieron recuperar porque tampoco tenían nombre_producto, por lo que no había forma de buscarlas (su tratamiento se explica en el siguiente cambio).
</div>  

In [22]:
orders = orders.dropna(subset=["categoria_producto"])

In [23]:
orders.shape

(24970, 12)

<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
    Se eliminaron las 30 filas restantes con nulos en nombre_producto y categoria_producto (0.12%, quedando 24,970 filas), ya que al no contar con el nombre del producto y por lo tanto, con su categoría (y sin forma de recuperarlos desde el catálogo), no se podría calcular su costo en el análisis de rentabilidad, y podría sesgar los datos en los pasos siguientes del análisis, alterando la visualización de los productos que aportan al monto total.
</div>  

In [24]:
print("Categorías únicas en Orders:",orders["categoria_producto"].unique())
print("Categorías únicas en Catalog:",catalog["categoria_producto"].unique())

Categorías únicas en Orders: ['Moda' 'Electrónica' 'Hogar']
Categorías únicas en Catalog: ['Electrónica' 'Hogar' 'Moda']


In [25]:
print("Valores Negativos en columna Cantidad:", (orders["cantidad"] < 0).sum())
orders[orders["cantidad"] < 0]

Valores Negativos en columna Cantidad: 4


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electrónica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electrónica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electrónica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electrónica,-1.0,423.53,0.0,-423.53


In [26]:
orders = orders[~(orders["cantidad"] < 0)]

In [27]:
print("Valores Negativos en columna Cantidad:", (orders["cantidad"] < 0).sum())

Valores Negativos en columna Cantidad: 0


<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
    Se detectaron 4 filas (0.02% del dataset) que tenían registro de valores negativos en cantidad y monto_total, se optó por eliminarlas para mantener limpios los registros de los montos positivos, ya que restarían revenue en el análisis de rentabilidad sin saber si son válidos, considerando que no se tiene definido por el área correspondiente, si representan devoluciones o alguna especificación que requiera mantenerlos. Pendiente: confirmar con esa área; el dataset quedó con 24,966 filas.
</div>  

In [28]:
print("Filas y columnas actualizadas:", orders.shape)
orders.describe()

Filas y columnas actualizadas: (24966, 12)


,cantidad,precio_unitario,monto_descuento,monto_total
count,24916.000000,24916.000000,24916.000000,2.496600e+04
mean,7.123214,259.367124,4.502930,2.081885e+03
std,297.072369,138.670945,5.224131,9.921507e+04
min,1.000000,20.030000,0.000000,5.240000e+00
25%,1.000000,138.535000,0.000000,1.807575e+02
50%,2.000000,258.775000,0.000000,3.419250e+02
75%,2.000000,380.355000,10.000000,5.185475e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [29]:
orders["cantidad"].value_counts().sort_index()

1.0        12330
2.0        12576
10000.0        6
20000.0        4
Name: cantidad, dtype: int64

In [30]:
orders[orders["cantidad"] > 2]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
3521,order_3521,user_5812,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electrónica,10000.0,43.14,0.0,431400.0
3522,order_3522,user_3575,2025-03-29,Argentina,desktop,social,Laptop-Gaming-16GB,Electrónica,10000.0,280.55,0.0,2805500.0
3586,order_3586,user_3380,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electrónica,10000.0,490.35,0.0,4903500.0
3643,order_3643,user_4440,2025-01-07,Colombia,desktop,social,Laptop-Gaming-16GB,Electrónica,10000.0,238.15,0.0,2381500.0
3656,order_3656,user_884,2025-01-01,Argentina,mobile,organic,Laptop-Gaming-16GB,Electrónica,20000.0,297.66,0.0,5953200.0
3668,order_3668,user_7270,2025-06-24,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electrónica,20000.0,348.31,0.0,6966200.0
3689,order_3689,user_6566,2025-06-16,mexico,desktop,paid_search,Laptop-Gaming-16GB,Electrónica,10000.0,87.69,0.0,876900.0
3722,order_3722,user_4723,2025-05-09,Argentina,mobile,paid_search,Laptop-Gaming-16GB,Electrónica,20000.0,442.01,0.0,8840200.0
3726,order_3726,user_2536,2025-02-12,Colombia,desktop,social,Laptop-Gaming-16GB,Electrónica,20000.0,290.85,0.0,5817000.0
3748,order_3748,user_7096,2025-02-23,Mexico,desktop,organic,Laptop-Gaming-16GB,Electrónica,10000.0,336.93,0.0,3369300.0


In [31]:
orders = orders[~(orders["cantidad"]>2)]

In [32]:
orders.shape

(24956, 12)

<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
    Al hacer la revisión estadística, se detectaron valores inusuales en 10 filas (0.04% del dataset), los cuales representaban compras en cantidades altas (mayores a 2, que representaba el rango máximo en el 75% de los datos), de 10 mil a 20 mil unidades en una sola compra, todas del producto Laptop-Gaming-16GB y con montos de hasta 8.8 millones, cuando el máximo normal rondaba los mil. Se decidió eliminar estos registros, dado que podrían influenciar negativamente en las gráficas y en el cálculo del revenue para los fines de este análisis. No se winsorizaron porque habría que inventar una cantidad que no existe en los datos. No se cuenta con la confirmación del área correspondiente sobre la veracidad de las compras o si han sido registros mal capturados. El dataset quedó con 24,956 filas.
</div>  

In [33]:
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,24906.000000,24906.000000,24906.000000,24956.000000
mean,1.504939,259.356605,4.504738,385.945141
std,0.499986,138.672693,5.224400,255.643812
min,1.000000,20.030000,0.000000,5.240000
25%,1.000000,138.525000,0.000000,180.670000
50%,2.000000,258.735000,0.000000,341.710000
75%,2.000000,380.365000,10.000000,517.680000
max,2.000000,499.960000,15.000000,999.890000


In [34]:
orders[orders["monto_total"] == 5.24]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
19686,order_19686,user_2326,2025-06-03,Mexico,desktop,social,Laptop-Gaming-16GB,Electrónica,1.0,20.24,15.0,5.24


In [35]:
orders["monto_calculado"] = orders["cantidad"] * orders["precio_unitario"] - orders["monto_descuento"]
print(orders.head())

  id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
0   order_0  user_6993        2025-05-22  Argentina     desktop   
1   order_1  user_1329        2025-06-15     Mexico     desktop   
2   order_2  user_3194        2025-05-02  Argentina     desktop   
3   order_3  user_4510        2025-06-09   Colombia      mobile   
4   order_4  user_5044        2025-03-30  Argentina     desktop   

  fuente_referencia       nombre_producto categoria_producto  cantidad  \
0           organic       Jacket-Winter-M               Moda       2.0   
1       paid_search  Tablet-Standard-64GB        Electrónica       1.0   
2            social        Blender-XL-Red              Hogar       2.0   
3            social  Tablet-Standard-64GB        Electrónica       1.0   
4       paid_search        Blender-XL-Red              Hogar       1.0   

   precio_unitario  monto_descuento  monto_total  monto_calculado  
0           332.69              0.0       665.37           665.38  
1           176.

In [36]:
orders["diferencia"] = (orders["monto_total"] - orders["monto_calculado"]).abs()
print(orders["diferencia"].describe())

count    2.490600e+04
mean     2.498996e-03
std      4.329634e-03
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      5.684342e-14
max      1.000000e-02
Name: diferencia, dtype: float64


In [37]:
orders[orders["monto_calculado"].isna()]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_calculado,diferencia
74,order_74,user_6172,2025-01-15,Argentina,desktop,organic,Sneakers-Urban-42,Moda,NaN,NaN,NaN,595.85,NaN,NaN
75,order_75,user_6588,2025-06-19,Colombia,mobile,organic,Sneakers-Urban-42,Moda,NaN,NaN,NaN,458.15,NaN,NaN
76,order_76,user_3193,2025-03-17,Argentina,mobile,paid_search,Laptop-Gaming-16GB,Electrónica,NaN,NaN,NaN,319.75,NaN,NaN
77,order_77,user_775,2025-06-24,Argentina,desktop,social,Vacuum-Pro-Black,Hogar,NaN,NaN,NaN,227.55,NaN,NaN
78,order_78,user_2702,2025-03-19,Argentina,desktop,paid_search,Phone-Pro-128GB,Electrónica,NaN,NaN,NaN,432.39,NaN,NaN
79,order_79,user_6438,2025-03-22,Colombia,desktop,paid_search,Sneakers-Urban-42,Moda,NaN,NaN,NaN,527.15,NaN,NaN
80,order_80,user_6790,2025-01-03,Mexico,desktop,organic,Vacuum-Pro-Black,Hogar,NaN,NaN,NaN,711.18,NaN,NaN
81,order_81,user_232,2025-05-16,Mexico,mobile,paid_search,Tablet-Standard-64GB,Electrónica,NaN,NaN,NaN,41.08,NaN,NaN
82,order_82,user_5665,2025-02-07,Colombia,mobile,paid_search,Sneakers-Urban-42,Moda,NaN,NaN,NaN,431.55,NaN,NaN
83,order_83,user_5225,2025-04-25,Argentina,mobile,organic,Sneakers-Urban-42,Moda,NaN,NaN,NaN,279.35,NaN,NaN


In [38]:
orders = orders.dropna(subset=["cantidad"])
print(orders.shape)

(24906, 14)


In [39]:
orders = orders.drop(["monto_calculado","diferencia"],axis=1)
print(orders.shape)

(24906, 12)


<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
    Se realizó la comprobación de que todas las cifras en monto_total de las filas completas correspondieran al patrón de multiplicar la cantidad × el precio_unitario, restando el monto del descuento. Se detectaron diferencias mínimas por el registro de decimales (máximo de 0.01, por redondeo a 2 decimales), sin notables alteraciones. Las filas con los nulos restantes en las 3 columnas fueron eliminadas (50 filas), ya que, a pesar de contar con el monto_total, no podríamos definir qué cantidad de productos se compraron y si se aplicaron descuentos, y al faltar los tres datos a la vez no se podían reconstruir con la fórmula; además, sin la cantidad no se puede calcular el costo en el análisis de rentabilidad. Las filas representan el 0.20% de los registros, conservando 24,906. (Se eliminan las columnas auxiliares de comprobación).
</div>  

In [40]:
orders.isna().sum()

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  296
dispositivo            20
fuente_referencia       0
nombre_producto         0
categoria_producto      0
cantidad                0
precio_unitario         0
monto_descuento         0
monto_total             0
dtype: int64

In [41]:
paises_por_usuario = orders.groupby("id_usuario")["pais"].nunique()
print((paises_por_usuario > 1).sum())

1997


In [42]:
print(orders["pais"].unique())
print(marketing["pais"].unique())

['Argentina' 'Mexico' 'Colombia' 'mexico' 'colombia' nan 'argentina']
['Mexico' 'Colombia' 'Argentina']


In [43]:
orders["pais"] = orders["pais"].str.strip().str.title()

In [44]:
orders["pais"].unique()

array(['Argentina', 'Mexico', 'Colombia', nan], dtype=object)

In [45]:
orders["pais"] = orders["pais"].fillna("Desconocido")

In [46]:
orders["pais"].value_counts()

Mexico         8308
Colombia       8280
Argentina      8022
Desconocido     296
Name: pais, dtype: int64

In [47]:
print(orders["pais"].unique())
print(marketing["pais"].unique())

['Argentina' 'Mexico' 'Colombia' 'Desconocido']
['Mexico' 'Colombia' 'Argentina']


<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
    Se unifican los formatos de los 3 países en las dos tablas. Al revisar los nulos de la columna país, se detectaron 296 (1.19% de los 24,906 registros). Para intentar recuperarlos con el país de cada usuario en compras anteriores, se revisó cuántos usuarios aparecían en más de un país: se detectó que 1,997 de 7,640 usuarios (26.14%) tienen registrado más de 1 país desde donde realizan la compra, por lo que no fue posible rellenar con el país de cada usuario en compras anteriores. Debido al % que los nulos representan, se mantuvieron los registros además de que cuentan con las otras columnas completas y se reemplazó por “Desconocido” para tenerlos en esa categoría e identificarlos fácilmente en los siguientes pasos y visualizaciones. 
    [Consecuencia: “Desconocido” no tiene gasto de marketing asociado, por lo que se tendrá en cuenta en el análisis de rentabilidad por país.]
</div>  

In [48]:
print("Nulos en columna Dispositivo:", orders["dispositivo"].isna().sum())
orders[orders["dispositivo"].isna()]

Nulos en columna Dispositivo: 20


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
24,order_24,user_458,2025-01-05,Colombia,NaN,organic,Sneakers-Urban-42,Moda,2.0,117.99,0.0,235.99
25,order_25,user_540,2025-03-08,Argentina,NaN,social,Vacuum-Pro-Black,Hogar,1.0,183.50,5.0,178.50
26,order_26,user_1951,2025-02-10,Colombia,NaN,organic,Blender-XL-Red,Hogar,1.0,331.52,15.0,316.52
27,order_27,user_5410,2025-05-09,Colombia,NaN,paid_search,Phone-Pro-128GB,Electrónica,2.0,449.48,5.0,893.96
28,order_28,user_1489,2025-01-17,Mexico,NaN,social,Phone-Pro-128GB,Electrónica,1.0,60.70,10.0,50.70
29,order_29,user_5197,2025-01-30,Colombia,NaN,social,Phone-Pro-128GB,Electrónica,1.0,381.57,0.0,381.57
30,order_30,user_5386,2025-01-31,Mexico,NaN,social,Vacuum-Pro-Black,Hogar,1.0,183.74,10.0,173.74
31,order_31,user_5766,2025-05-05,Colombia,NaN,paid_search,Vacuum-Pro-Black,Hogar,1.0,480.74,0.0,480.74
32,order_32,user_4400,2025-03-29,Mexico,NaN,social,Tablet-Standard-64GB,Electrónica,1.0,117.69,0.0,117.69
33,order_33,user_5070,2025-01-12,Argentina,NaN,paid_search,Tablet-Standard-64GB,Electrónica,1.0,96.82,10.0,86.82


In [49]:
orders["id_usuario"].nunique()

7640

In [50]:
orders["dispositivo"] = orders["dispositivo"].fillna("Desconocido")
print(orders["dispositivo"].value_counts())

desktop        12661
mobile         12225
Desconocido       20
Name: dispositivo, dtype: int64


<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
    Se conservan los 20 nulos de la columna dispositivo (0.08% de los registros), toda vez que los datos de las demás columnas están completos y no es posible inferir el dispositivo desde otras columnas. Se crea la categoría “Desconocido” para fines de los siguientes pasos, para que no se pierdan estos pedidos ni se confundan con desktop o mobile. El efecto en el análisis es mínimo: solo agrega una categoría extra al comparar dispositivos.
</div>  

In [51]:
orders["fecha_hora_pedido"]= pd.to_datetime(orders["fecha_hora_pedido"])
marketing["fecha"] = pd.to_datetime(marketing["fecha"])
print(orders["fecha_hora_pedido"].dtype)
print(marketing["fecha"].dtype)

datetime64[ns]
datetime64[ns]


In [52]:
orders["cantidad"] = orders["cantidad"].astype("int64")
print(orders["cantidad"].dtypes)

int64



<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
    Se realiza la conversión de formato de las columnas que contienen fechas en dos tablas (fecha_hora_pedido en Orders y fecha en Marketing), a formato fecha (datetime64) con pd.to_datetime(), ya que estaban como texto y así se podrán ordenar por tiempo, agrupar por periodos y cruzar entre tablas; así como la conversión a número entero (int64) de la columna cantidad de la tabla Orders, que estaba como decimal (float64) por los nulos; al eliminarlos ya se pudo convertir, y las unidades vendidas son siempre enteras.
</div>
  

In [53]:
marketing["canal"].value_counts()

paid_search    507
social         506
organic        506
Name: canal, dtype: int64

In [54]:
marketing[marketing["canal"].isna()]

,fecha,pais,id_campaña,canal,gasto
98,2025-01-11,Argentina,social_Argentina,NaN,849.70
99,2025-01-12,Mexico,organic_Mexico,NaN,2033.56
100,2025-01-12,Mexico,paid_search_Mexico,NaN,1260.65
101,2025-01-12,Mexico,social_Mexico,NaN,1660.90
102,2025-01-12,Colombia,organic_Colombia,NaN,1819.27
...,...,...,...,...,...
194,2025-01-22,Colombia,social_Colombia,NaN,772.46
195,2025-01-22,Argentina,organic_Argentina,NaN,1562.61
196,2025-01-22,Argentina,paid_search_Argentina,NaN,644.34
197,2025-01-22,Argentina,social_Argentina,NaN,2544.66


In [55]:
marketing["canal_prueba"] = (marketing["canal"] + "_" + marketing["pais"])
print(marketing.head())
print((marketing["canal_prueba"] != marketing["id_campaña"]).sum())

       fecha      pais            id_campaña        canal    gasto  \
0 2025-01-01    Mexico        organic_Mexico      organic  2446.25   
1 2025-01-01    Mexico    paid_search_Mexico  paid_search  2704.34   
2 2025-01-01    Mexico         social_Mexico       social  2045.01   
3 2025-01-01  Colombia      organic_Colombia      organic  2597.21   
4 2025-01-01  Colombia  paid_search_Colombia  paid_search  1771.40   

           canal_prueba  
0        organic_Mexico  
1    paid_search_Mexico  
2         social_Mexico  
3      organic_Colombia  
4  paid_search_Colombia  
101


In [56]:
marketing["id_campaña"].str.rsplit("_", n=1).str[0].value_counts()

social         540
paid_search    540
organic        540
Name: id_campaña, dtype: int64

In [57]:
marketing["canal"] = marketing["id_campaña"].str.rsplit("_", n=1).str[0]
print(marketing["canal"].isna().sum())

0


In [58]:
marketing = marketing.drop(["canal_prueba"],axis=1)
print(marketing.columns)

Index(['fecha', 'pais', 'id_campaña', 'canal', 'gasto'], dtype='object')


In [59]:
print("Duplicados en marketing:", marketing.duplicated().sum())
print("Duplicados en catalog:", catalog.duplicated().sum())

Duplicados en marketing: 0
Duplicados en catalog: 0


In [60]:
print(marketing.duplicated(subset=["fecha", "pais", "canal"]).sum())
print(marketing[marketing.duplicated(subset=["fecha", "pais", "canal"], keep=False)].sort_values(by=["fecha", "pais", "canal"]).head(10))

0
Empty DataFrame
Columns: [fecha, pais, id_campaña, canal, gasto]
Index: []


In [61]:
print(orders["fuente_referencia"].unique())
print(marketing["canal"].unique())

['organic' 'paid_search' 'social']
['organic' 'paid_search' 'social']




<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
    Se encontraron 101 nulos (6.2% de las 1,620 filas) en la columna canal, de la tabla marketing. Al hacer una comparación con la columna id_campaña, se determinó crear una columna auxiliar (canal_prueba) que une canal, un guion bajo y país, para confirmar que la regla se cumple: en las 1,519 filas que sí tenían canal, el id_campaña coincidía siempre, y las únicas 101 diferencias eran las filas vacías. Con eso se concluyó que los nulos podrían rellenarse extrayendo la primera parte del contenido de las celdas, antes del país (cortando desde el último guion bajo, porque canales como paid_search ya traen uno). La extracción efectivamente logró completar los datos faltantes, retornando 0 nulos en la columna canal, con solo 3 canales sin variantes de escritura (organic, paid_search y social). Se eliminó la columna auxiliar y se verificó que no hay duplicados por fecha, país y canal.
</div>  



In [62]:
print(sorted(orders["nombre_producto"].unique()))
print(sorted(catalog["nombre_producto"].unique()))

['Blender-XL-Red', 'Jacket-Winter-M', 'Laptop-Gaming-16GB', 'Phone-Pro-128GB', 'Sneakers-Urban-42', 'Tablet-Standard-64GB', 'Vacuum-Pro-Black']
['Blender-XL-Red', 'Jacket-Winter-M', 'Laptop-Gaming-16GB', 'Phone-Pro-128GB', 'Sneakers-Urban-42', 'Tablet-Standard-64GB', 'Vacuum-Pro-Black']


<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>💡 Validaciones sin cambios:</b>
<ul>
<li>La columna monto_total corresponde a lo esperado en todas las filas que tienen las tres columnas de la fórmula: cantidad, precio_unitario y monto_descuento (diferencia máxima de 0.01, por redondeo).</li>
<li>Las tablas Marketing y Catalog no cuentan con duplicados (en Marketing, también por fecha, país y canal).</li>
<li>Los nombre_producto de las tablas Orders y Catalog son consistentes (los mismos 7 productos), por lo que todas las ventas tendrán un costo.</li>
<li>La venta con monto_total de 5.24 es correcta, considerando la aplicación del descuento (el mínimo se revisó por parecer bajo y cuadra con un precio cercano al mínimo y descuento de 15).</li>
</ul>
</div>

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [63]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 


In [64]:
revenue = orders["monto_total"].sum().round(2)
print("Ingreso Total:", revenue)

Ingreso Total: 9610018.94


In [65]:
orders_costo = pd.merge(orders,catalog[["nombre_producto","costo_unitario"]],on=["nombre_producto"], how="left")
print(orders_costo.shape)

(24906, 13)


In [66]:
orders_costo["costo_unitario"].isna().sum()

0

In [67]:
orders_costo["costo_pedido"] = (orders_costo["cantidad"] * orders_costo["costo_unitario"])
print(orders_costo.head())

  id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
0   order_0  user_6993        2025-05-22  Argentina     desktop   
1   order_1  user_1329        2025-06-15     Mexico     desktop   
2   order_2  user_3194        2025-05-02  Argentina     desktop   
3   order_3  user_4510        2025-06-09   Colombia      mobile   
4   order_4  user_5044        2025-03-30  Argentina     desktop   

  fuente_referencia       nombre_producto categoria_producto  cantidad  \
0           organic       Jacket-Winter-M               Moda         2   
1       paid_search  Tablet-Standard-64GB        Electrónica         1   
2            social        Blender-XL-Red              Hogar         2   
3            social  Tablet-Standard-64GB        Electrónica         1   
4       paid_search        Blender-XL-Red              Hogar         1   

   precio_unitario  monto_descuento  monto_total  costo_unitario  costo_pedido  
0           332.69              0.0       665.37          189.31       

---

In [68]:
costo_total = orders_costo["costo_pedido"].sum().round(2)
print("Costo Total:", costo_total)

Costo Total: 3828869.01


<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
    Para el cálculo del costo_total, se unió toda la tabla Orders, con las columnas nombre_producto y costo_unitario de la tabla Catalog con .merge(), se usó el tipo de unión Left, debido a que se necesitan mantener todos los datos de la tabla Orders y agregar únicamente esas dos columnas de la otra tabla, para evitar columnas duplicadas y obtener la extracción del costo_unitario de cada producto (se verificó que no hubo nulos, es decir, todos los pedidos tienen costo), multiplicando por cantidad comprada para calcular el costo por pedido, seguido de realizar el cálculo del Costo Total para obtener las siguientes métricas de negocio.
</div>  

In [69]:
marketing_total = marketing["gasto"].sum().round(2)
print("Inversión en Marketing:", marketing_total)

Inversión en Marketing: 2871843.53


In [70]:
profit = revenue - costo_total - marketing_total
margen_profit = profit / revenue * 100
print("Profit:", profit)
print("Margen de Ganancia (%):", margen_profit)


Profit: 2909306.4
Margen de Ganancia (%): 30.27368018902156


In [71]:
pedidos_con_perdida = (orders_costo["monto_total"]<orders_costo["costo_pedido"]).sum()
pct_pedidos_con_perdida = (pedidos_con_perdida / orders_costo.shape[0] * 100).round(2)
print("Filas con Ingreso menor al Costo:", pedidos_con_perdida)
print("Porcentaje de pedidos con pérdida:", pct_pedidos_con_perdida)
orders_costo[orders_costo["monto_total"]<orders_costo["costo_pedido"]].head(10)

Filas con Ingreso menor al Costo: 4528
Porcentaje de pedidos con pérdida: 18.18


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario,costo_pedido
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2,102.99,10.0,195.99,176.64,353.28
6,order_6,user_2802,2025-03-15,Colombia,mobile,organic,Blender-XL-Red,Hogar,2,163.59,0.0,327.19,176.64,353.28
19,order_19,user_7190,2025-02-03,Argentina,mobile,social,Jacket-Winter-M,Moda,2,45.86,0.0,91.72,189.31,378.62
39,order_39,user_7855,2025-01-29,Colombia,Desconocido,paid_search,Blender-XL-Red,Hogar,1,58.16,0.0,58.16,176.64,176.64
41,order_41,user_2963,2025-01-27,Mexico,Desconocido,social,Jacket-Winter-M,Moda,2,37.77,10.0,65.53,189.31,378.62
51,order_131,user_2914,2025-05-05,Desconocido,mobile,organic,Blender-XL-Red,Hogar,1,117.84,10.0,107.84,176.64,176.64
60,order_140,user_2308,2025-04-05,Desconocido,desktop,social,Jacket-Winter-M,Moda,1,107.28,0.0,107.28,189.31,189.31
64,order_144,user_5041,2025-05-05,Desconocido,mobile,organic,Blender-XL-Red,Hogar,2,177.04,10.0,344.08,176.64,353.28
82,order_162,user_4209,2025-04-12,Desconocido,mobile,social,Vacuum-Pro-Black,Hogar,1,27.70,15.0,12.70,16.60,16.60
91,order_171,user_3243,2025-06-08,Desconocido,desktop,paid_search,Blender-XL-Red,Hogar,2,155.61,10.0,301.23,176.64,353.28


In [72]:
filas_con_perdida = orders_costo[orders_costo["monto_total"] < orders_costo["costo_pedido"]]
filas_con_perdida["nombre_producto"].value_counts()

Jacket-Winter-M         1525
Laptop-Gaming-16GB      1498
Blender-XL-Red          1414
Tablet-Standard-64GB      48
Sneakers-Urban-42         31
Vacuum-Pro-Black          10
Phone-Pro-128GB            2
Name: nombre_producto, dtype: int64

<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
Se observa que el costo unitario de cada producto no cambia, sin embargo, el precio unitario sí tiene fluctuaciones que no sabemos por qué suceden, además de los descuentos que se aplican tampoco sabemos de qué dependen. Como el costo es fijo y el precio varía mucho (de 20 a casi 500), un pedido pierde dinero cuando su precio cae por debajo del costo del producto, y eso ocurre sobre todo en los tres productos con costo más alto (Jacket-Winter-M, Laptop-Gaming-16GB y Blender-XL-Red), que concentran el X% de las pérdidas. Los descuentos son pequeños (máximo 15), por lo que el principal factor es el precio. Se recomienda preguntar al área correspondiente por qué varía el precio y cómo se definen los descuentos.
</div>

In [73]:
ticket_promedio = orders["monto_total"].mean().round(2)
print("Ticket Promedio:", ticket_promedio)

Ticket Promedio: 385.85


In [74]:
print("Cantidad Promedio de Productos por Orden:", orders["cantidad"].mean().round(2))

Cantidad Promedio de Productos por Orden: 1.5


In [75]:
producto_mas_vendido = orders.groupby("nombre_producto")["cantidad"].sum()
print(producto_mas_vendido.sort_values(ascending=False).head(1))

nombre_producto
Vacuum-Pro-Black    6284
Name: cantidad, dtype: int64


In [76]:
gasto_marketing_por_canal = marketing.groupby("canal")["gasto"].sum().sort_values(ascending=False)
print("Gasto Total en Marketing:", marketing_total)
print()
print(gasto_marketing_por_canal)

Gasto Total en Marketing: 2871843.53

canal
social         976818.37
organic        972650.96
paid_search    922374.20
Name: gasto, dtype: float64


<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>💡 CONCLUSIONES:</b>
Con los datos disponibles, el negocio es rentable. El ingreso total fue de 9,610,018.94, con un costo de productos de 3,828,869.01 y una inversión en marketing de 2,871,843.53, lo que deja un profit de 2,909,306.40 y un margen de ganancia de 30.27%. Del ingreso, aproximadamente 39.8% se va en costo de productos, 29.9% en marketing y 30.3% queda como ganancia.

El ticket promedio por orden es de 385.85, con 1.5 productos por orden. El producto más vendido por unidades es Vacuum-Pro-Black (6,284), y el gasto de marketing se reparte de forma casi pareja entre los canales social (976,818.37), organic (972,650.96) y paid_search (922,374.20).

Hallazgo principal: aunque el negocio gana en total, 4,528 pedidos (18.18%) se vendieron con un ingreso menor a su costo. El 97.99% de ellos corresponde a tres productos: Jacket-Winter-M, Laptop-Gaming-16GB y Blender-XL-Red. El costo de cada producto es fijo, pero el precio de venta varía mucho entre pedidos, y los descuentos (máximo 15) no alcanzan a explicarlo.

Pendientes para el área de negocio: confirmar por qué varía el precio unitario y cómo se definen los descuentos, validar los registros excluidos en la limpieza (4 con valores negativos y 10 con cantidades de 10 mil a 20 mil unidades) y aclarar por qué el canal organic registra gasto de marketing casi igual al de los pagados. Este cálculo de profit solo considera costo de productos y marketing, sin otros costos operativos ni ingresos por suscripción, que no están en los datos.
</div> 

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 
Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?


---

In [77]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (credenciales omitidas por seguridad)
# ======================
db_config = {
    'user': 'Tu_usuario',
    'pwd': 'Tu_contrasena',
    'host': 'Tu_host',
    'port': 'Tu_puerto',
    'db': 'Tu_base_de_datos'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [78]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [79]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   id_usuario          120000 non-null  object
 1   id_sesion           120000 non-null  object
 2   nombre_evento       120000 non-null  object
 3   timestamp_evento    120000 non-null  object
 4   pais                120000 non-null  object
 5   dispositivo         120000 non-null  object
 6   fuente_referencia   120000 non-null  object
 7   categoria_producto  120000 non-null  object
dtypes: object(8)
memory usage: 7.3+ MB


In [80]:
events["timestamp_evento"] = pd.to_datetime(events["timestamp_evento"])
print(events["timestamp_evento"].dtypes)

datetime64[ns]


In [81]:
events.duplicated().sum()

0

In [82]:
events.duplicated(subset=["id_usuario", "id_sesion", "nombre_evento", "timestamp_evento"]).sum()

0

In [83]:
events["nombre_evento"].unique().tolist()

['first_visit',
 'add_to_cart',
 'add_payment_info',
 'purchase',
 'begin_checkout',
 'select_item']

In [84]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS total_usuarios
FROM events
GROUP BY nombre_evento
ORDER BY CASE nombre_evento
    WHEN 'first_visit' THEN 1
    WHEN 'select_item' THEN 2
    WHEN 'add_to_cart' THEN 3
    WHEN 'begin_checkout' THEN 4
    WHEN 'add_payment_info' THEN 5
    WHEN 'purchase' THEN 6
END
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,total_usuarios
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [85]:
query_usuarios_sin_select = '''
SELECT COUNT(DISTINCT id_usuario) AS usuarios_sin_select
FROM events
WHERE nombre_evento = 'add_to_cart'
  AND id_usuario NOT IN (
      SELECT id_usuario
      FROM events
      WHERE nombre_evento = 'select_item'
  )
'''

usuarios_sin_select_item = pd.read_sql(query_usuarios_sin_select, con=engine)
usuarios_sin_select_item

,usuarios_sin_select
0,401


<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
Se utiliza una subconsulta con NOT IN para excluir a los usuarios que tienen al menos un registro del evento select_item. De esta manera, se cuentan únicamente los usuarios únicos que realizaron add_to_cart sin tener registrado el evento de selección. El resultado fueron 401 usuarios, que representan el 5.25% de los 7,634 usuarios que llegaron al carrito. Esta cifra es mayor que la diferencia de 52 usuarios entre select_item y add_to_cart porque esa diferencia es un resultado neto: los 401 usuarios que llegaron al carrito sin select_item se compensan en gran parte con usuarios que sí seleccionaron producto pero no llegaron al carrito. Una posible explicación es que parte de estos usuarios agregó productos directamente al carrito desde un listado, mediante un botón de agregar, o llegó desde una campaña directo a la página del producto, sin pasar por select_item. El resultado permite identificar posibles comportamientos atípicos en el recorrido de compra o eventos que no quedaron registrados. Nota: no podemos asegurar que nunca seleccionaron un producto; únicamente sabemos que ese evento no aparece en los datos.
</div>

In [86]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS total_usuarios,
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'select_item' THEN 2
            WHEN 'add_to_cart' THEN 3
            WHEN 'begin_checkout' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
        END AS orden_funnel
    FROM events
    GROUP BY nombre_evento
),

conversiones AS (
    SELECT
        nombre_evento,
        total_usuarios,
        orden_funnel,
        LAG(total_usuarios) OVER (
            ORDER BY orden_funnel
        ) AS usuarios_anteriores
    FROM funnel
)

SELECT
    nombre_evento,
    usuarios_anteriores,
    total_usuarios,
    usuarios_anteriores - total_usuarios AS usuarios_perdidos,
    ROUND(total_usuarios * 100.0 / NULLIF(usuarios_anteriores, 0), 2) AS conversion_pct,
    ROUND(100.0 - (total_usuarios * 100.0 / NULLIF(usuarios_anteriores, 0)), 2) AS drop_off_pct
FROM conversiones
WHERE usuarios_anteriores IS NOT NULL
ORDER BY usuarios_perdidos DESC;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_anteriores,total_usuarios,usuarios_perdidos,conversion_pct,drop_off_pct
0,add_payment_info,7208,6250,958,86.71,13.29
1,begin_checkout,7634,7208,426,94.42,5.58
2,select_item,7796,7582,214,97.26,2.74
3,purchase,6250,6240,10,99.84,0.16
4,add_to_cart,7582,7634,-52,100.69,-0.69


In [87]:
query_conversion_global = '''
WITH conteos AS (
    SELECT
        COUNT(DISTINCT CASE WHEN nombre_evento = 'first_visit' THEN id_usuario END) AS usuarios_first_visit,
        COUNT(DISTINCT CASE WHEN nombre_evento = 'purchase' THEN id_usuario END) AS usuarios_purchase
    FROM events)

SELECT
    usuarios_first_visit,
    usuarios_purchase,
    ROUND(usuarios_purchase * 100.0 / NULLIF(usuarios_first_visit, 0), 2) AS conversion_global_pct
FROM conteos;
'''

conversion_global = pd.read_sql(query_conversion_global, con=engine)
conversion_global

,usuarios_first_visit,usuarios_purchase,conversion_global_pct
0,7796,6240,80.04


<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
El 80.04% de los usuarios con first_visit (7,796) registró al menos un purchase (6,240).
</div>

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [88]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [89]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_usuario      8000 non-null   object
 1   fecha_registro  8000 non-null   object
 2   país            8000 non-null   object
 3   dispositivo     8000 non-null   object
 4   tipo_plan       8000 non-null   object
dtypes: object(5)
memory usage: 312.6+ KB


In [90]:
users["id_usuario"].nunique()

8000

In [91]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(5)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0
4,user_1,2025-01-14,7,0


In [92]:
user_activity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000 entries, 0 to 31999
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_usuario             32000 non-null  object
 1   fecha_actividad        32000 non-null  object
 2   dias_despues_registro  32000 non-null  int64 
 3   activo                 32000 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 1000.1+ KB


In [93]:
#Identificación de cohortes

query_mes_cohorte = '''
SELECT
    id_usuario,
    DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS mes_cohorte
FROM users;
'''

# Ejecutar la consulta
mes_cohorte = pd.read_sql(query_mes_cohorte, con=engine)
mes_cohorte

,id_usuario,mes_cohorte
0,user_0,2025-01-01 00:00:00+00:00
1,user_1,2025-01-01 00:00:00+00:00
2,user_2,2025-03-01 00:00:00+00:00
3,user_3,2025-03-01 00:00:00+00:00
4,user_4,2025-02-01 00:00:00+00:00
...,...,...
7995,user_7995,2025-05-01 00:00:00+00:00
7996,user_7996,2025-01-01 00:00:00+00:00
7997,user_7997,2025-04-01 00:00:00+00:00
7998,user_7998,2025-05-01 00:00:00+00:00


In [94]:
#Identificación total de usuarios por cohorte

query_total_usuarios = '''
SELECT
    DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS mes_cohorte,
    COUNT(DISTINCT id_usuario) AS total_usuarios
FROM users
GROUP BY mes_cohorte
ORDER BY mes_cohorte;
'''

# Ejecutar la consulta
total_usuarios = pd.read_sql(query_total_usuarios, con=engine)
total_usuarios

,mes_cohorte,total_usuarios
0,2025-01-01 00:00:00+00:00,1627
1,2025-02-01 00:00:00+00:00,1444
2,2025-03-01 00:00:00+00:00,1636
3,2025-04-01 00:00:00+00:00,1606
4,2025-05-01 00:00:00+00:00,1687


In [95]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH retencion_porcentaje AS (
    SELECT
        DATE_TRUNC('month', CAST(u.fecha_registro AS DATE)) AS mes_cohorte,
        COUNT(DISTINCT u.id_usuario) AS total_usuarios,

        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 7 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w1,

        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 14 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w2,

        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 21 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w3

    FROM users AS u
    LEFT JOIN user_activity AS a
        ON u.id_usuario = a.id_usuario
    GROUP BY mes_cohorte
)

SELECT
    mes_cohorte,
    total_usuarios,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    ROUND(retenido_w1 * 100.0 / total_usuarios, 2) AS semana_1,
    ROUND(retenido_w2 * 100.0 / total_usuarios, 2) AS semana_2,
    ROUND(retenido_w3 * 100.0 / total_usuarios, 2) AS semana_3
FROM retencion_porcentaje
ORDER BY mes_cohorte;
'''

cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,mes_cohorte,total_usuarios,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 Comentario:</b>
La retención semanal es estable entre las cohortes, con valores entre 40.07% y 43.98%, y no disminuye de forma marcada de la semana 1 a la 3. La cohorte de febrero muestra el valor más alto (43.98% en la semana 3); una posible explicación sería alguna promoción en ese mes, aunque no podemos confirmarlo con estos datos. En general, menos de la mitad de los usuarios sigue activo cada semana.
</div>

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** No hay diferencia en la proporción de usuarios que compran, entre los usuarios del grupo de p_control y los que lo hicieron mediante p_tratamiento. Ambas formas arrojan el mismo resultado.
   - **H₁ (Hipótesis alternativa):** La proporción de usuarios que compra en el grupo de tratamiento es distinta a la del grupo de control (p_tratamiento ≠ p_control).
   
**Test estadístico:** Z de Proporciones (Z-test)

**Nivel de significancia alpha:** 0.05 

In [96]:
# tu código aquí
experiment_checkout_ui = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv")

In [97]:
experiment_checkout_ui.info()
experiment_checkout_ui.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [98]:
experiment_checkout_ui.describe()

,convirtio,duracion_sesion
count,10000.000000,10000.000000
mean,0.159900,159.862439
std,0.366532,81.074410
min,0.000000,20.010000
25%,0.000000,89.470000
50%,0.000000,159.725000
75%,0.000000,229.745000
max,1.000000,300.000000


In [99]:
experiment_checkout_ui[experiment_checkout_ui.duplicated(keep=False)].sort_values(by="id_usuario").head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp


In [100]:
experiment_checkout_ui["variante"].unique()

array(['tratamiento', 'control'], dtype=object)

In [101]:
experiment_checkout_ui["variante"].value_counts()

tratamiento    5035
control        4965
Name: variante, dtype: int64

In [102]:
experiment_checkout_ui.groupby("variante")["convirtio"].agg(['sum', 'count', 'mean'])

,sum,count,mean
variante,,,
control,779,4965,0.156898
tratamiento,820,5035,0.162860


In [103]:
from statsmodels.stats.proportion import proportions_ztest

---

In [104]:
conversiones = experiment_checkout_ui.groupby("variante")["convirtio"].sum()
conversiones

variante
control        779
tratamiento    820
Name: convirtio, dtype: int64

In [105]:
totales = experiment_checkout_ui.groupby("variante")["convirtio"].count()
totales

variante
control        4965
tratamiento    5035
Name: convirtio, dtype: int64

In [106]:
exitos = [conversiones["control"], conversiones["tratamiento"]]
exitos

[779, 820]

In [107]:
observaciones = [totales["control"], totales["tratamiento"]]
observaciones

[4965, 5035]

In [108]:
z_stat, p_value = proportions_ztest(exitos, observaciones)
print(f"Estadístico Z: {z_stat}")
print(f"Valor p: {p_value}")
alpha = 0.05
if p_value < alpha:
    print("Rechazamos la hipóstesis nula: hay evidencia de una diferencia")
else:
    print("No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia")

Estadístico Z: -0.8132782986429474
Valor p: 0.41605851639119995
No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia


<div style="background-color: rgba(100, 170, 220, 0.30); padding: 10px; border-radius: 5px; color: black;">
<b>📝 </b>
    
**Interpretación:** La tasa de conversión del grupo de control fue de 15.69%, mientras que la del grupo con la modificación de UI fue de 16.29%, lo que representa una diferencia de 0.60 puntos porcentuales. Sin embargo, la prueba Z obtuvo un p-value de 0.416, superior al nivel de significancia establecido (α = 0.05). Por lo tanto, no rechazamos H₀, ya que no existe evidencia estadística suficiente para afirmar que la modificación haya mejorado la tasa de conversión.
    
**Recomendación:** No se recomienda implementar la modificación basándose únicamente en estos resultados. Dado que la diferencia observada es pequeña (0.60 puntos porcentuales), se sugiere realizar un nuevo experimento con una muestra mayor para evaluar si existe un efecto real.

**Limitación:** El resultado no demuestra que la modificación carezca de efecto, sino que este experimento no encontró evidencia suficiente para afirmarlo. Es posible que la muestra no tuviera el tamaño necesario para detectar un efecto pequeño.
</div>

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc


---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [109]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive